# Notebook 41 - Do compressed detectors forget attack families they never saw?

**Why this notebook.** Notebook 40 showed that on these detectors, removing up to 90% of the computation costs nothing in deployed risk once the pruned model is recovered with two full epochs. A pruning method has nothing to win in that setting. The one place where recovery cannot help is traffic that is not in the training data: attack families the detector never saw. Compressed models are known to keep their aggregate accuracy while losing rare and atypical inputs. If that happens here, the dense model catches an unseen family more often than its pruned copy, recovery on training data cannot close that gap, and a method that preserves off-distribution behaviour during recovery has a real target. This notebook measures the gap before any method is designed.

**What it does.** Three attack families are held out in turn (DoS, reconnaissance, web application). For each one, a dense teacher is trained from scratch without that family, pruned to 70% and 90% realised FLOP reduction with two methods (magnitude and Fisher), and recovered with two full epochs, still without that family. The reference is the same dense teacher given the same two extra epochs under the same seed, so every comparison is paired. Both networks, five seeds. On the held-out family the question is simple: what fraction of its flows does the model still raise as an attack, and where does the rest go?

**Readings, fixed in stage 2 before any result.** P1, the gap: at 90%, for at least two of the three families and both methods, the pruned detector catches at least 5 percentage points less of the unseen family than the dense reference (paired 95% interval above zero), while its in-distribution risk is matched within 0.02. Secondary: whether the gap grows from 70% to 90%. The four readings and what each means for a method are written in the pre-registration.

**Stages.** 1 bootstrap, 2 pre-registration, 2b commit of the pre-registration, 3 data and helpers, 4 the runs (30 cells, each one teacher + one dense reference + four students, resumable, teachers cached under models/), 5 analysis and verdict, 6 figure, 7 commit of the results. GPU; about four hours. If the session stops, rerun from the top: finished cells are skipped.

In [ ]:
# Stage 1 - bootstrap
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os, sys, json, copy, itertools, gc, time, subprocess
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn, yaml
import matplotlib.pyplot as plt
from scipy import stats

REPO = Path("/content/drive/MyDrive/SABER_Research/saber-ids")
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from src.saber.bridge_ciciot import load_bridge
from src.saber.deep_model import DeepCNN1D
from src import models as M
from src.saber.taxonomy import ciciot2023_taxonomy, DEFAULT_COST_PROFILES
from src.saber.metrics import full_model_audit
from src.saber.surgery import enumerate_cnn1d_channel_groups, prune_cnn1d_channels, profile_forward_flops, count_parameters
from src.saber.leverage import magnitude_scores, gradient_saliency_scores
from src.saber.postg5 import calibrate_prefix_count

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
R = REPO / "results/saber"; OUT = R / "41_unseen_family_compression"; OUT.mkdir(parents=True, exist_ok=True)
TEACHER_DIR = REPO / "models/ciciot2023/lofo"; TEACHER_DIR.mkdir(parents=True, exist_ok=True)   # git-ignored
print("device:", DEVICE)

In [ ]:
# Stage 2 - pre-registration (frozen: rerunning this cell can never change it after its commit)
PREREG = {
    "notebook": "41_unseen_family_compression",
    "question": ("does a pruned and recovered CICIoT2023 detector raise an attack family it never saw in training less often than "
                 "the dense detector trained and recovered the same way, at matched in-distribution risk"),
    "motivation": ("notebook 40: up to 90% realised FLOP reduction costs nothing in deployed risk after two full epochs of recovery, so "
                   "no pruning method has anything to win in distribution; recovery cannot help on traffic absent from the training "
                   "data, and compressed models are known to lose rare and atypical inputs while keeping aggregate accuracy; the "
                   "locked test partition already showed the shallow students losing 0.06-0.13 fine macro-F1 against the teacher's "
                   "0.03; this notebook measures the unseen-family gap before any method is designed"),
    "design": {
        "corpus": "CICIoT2023, frozen split, validation only (no test access); evaluation subsample of notebooks 37-40 (up to 4,000 rows per class, seed 0)",
        "held_out_families": ["dos", "reconnaissance", "web_application"],
        "held_out_rule": ("every class of the held-out family is removed from the training rows, the class weights, the clipping "
                          "bounds, the recalibration slice and the Fisher batches; the models have one output per remaining class; the "
                          "in-distribution audit uses the evaluation rows of the remaining classes; the unseen audit uses the evaluation "
                          "rows of the held-out family; one thing is shared across settings and not refit per family: the frozen "
                          "bridge's per-feature standardisation (mean and standard deviation of the full training split), which carries "
                          "no label information"),
        "architectures": "shallow CNN (64/128) and deep CNN (64/128/128/256), one output per remaining class",
        "teacher": ("trained from scratch per (architecture, family, seed): 4 epochs, Adam 1e-3, batch 1,024, inverse-square-root "
                    "class weights normalised to unit mean, seeded initialisation and shuffle, inputs clipped at the 0.1 and 99.9 "
                    "percentiles of 400,000 remaining training rows (seed 2026)"),
        "teacher_acceptance": ("on the remaining classes' evaluation rows, after recalibration: benign escalation at most 0.10, attack "
                               "recall at least 0.85, fine macro-F1 at least 0.40 (looser than notebook 29's 0.90 and 0.50 because "
                               "holding out a family removes some of the easiest classes from the pooled recall and the mean); the rule "
                               "exists to exclude a broken teacher, not to select a good one; a cell whose teacher fails is excluded "
                               "from the verdict and listed"),
        "dense_reference": "the cell's teacher given two further epochs under the cell's recovery seed, then recalibrated (same treatment as the students)",
        "students": ("magnitude (L1) and Fisher scores computed on the teacher (Fisher on 40 training batches of remaining classes); "
                     "removal orders ascending with per-layer floors (shallow 4, deep 8); prefixes calibrated to 70% and 90% "
                     "realised FLOP reduction (tolerance 1.5 points); recovered two full epochs over the remaining training rows under "
                     "the cell's recovery seed; recalibrated"),
        "recalibration": "BatchNorm running statistics re-estimated on the first 51,200 rows of a seed-2026 permutation of the remaining training rows, weights frozen",
        "seeds": [101, 211, 307, 401, 503],
        "cells": "2 architectures x 3 families x 5 seeds = 30 teachers, 30 dense references, 120 students",
    },
    "endpoints": {
        "unseen_detect": "fraction of the held-out family's evaluation rows predicted as any attack class (argmax rule), the primary endpoint",
        "unseen_absorption": "share of the held-out family's rows predicted as benign and as each remaining attack family (descriptive)",
        "unseen_attack_mass": "mean of 1 minus the benign probability on the held-out rows (threshold-free, descriptive)",
        "in_distribution": "recalibrated ground-truth HSR (balanced profile), benign escalation, attack-miss rate, family and fine macro-F1, ECE and per-family recall on the remaining classes",
        "paired_gap": "dense reference's unseen_detect minus the student's, under the same seed; mean over accepted seeds with a 95% t interval",
        "matched": "the mean paired difference in in-distribution HSR between student and dense reference is within 0.02",
    },
    "P1_gap": ("for an architecture, P1 holds if at 90% realised reduction, for at least two of the three held-out families and for "
               "both methods, the paired gap in unseen_detect is at least 0.05 with the lower end of its 95% interval above zero, and "
               "the cell is matched in distribution"),
    "S1_dose": "descriptive: in the majority of (architecture, family, method) cells the paired gap at 90% exceeds the gap at 70%",
    "S2_ceiling": "descriptive: the dense reference's own unseen_detect per family, the ceiling against which the gap is read",
    "readings": {
        "per_architecture": ("gap_at_matched_risk if P1 holds; gap_but_unmatched if the gap condition holds in at least two families for "
                             "both methods but matching fails in one of those cells; partial if at least one cell meets the gap "
                             "condition but P1 fails; no_gap if no cell meets it; a separate flag student_better is set when at least "
                             "half of the 90% cells have a negative gap with the upper end of the interval below zero"),
        "overall": ("route_open if both architectures read gap_at_matched_risk; route_open_one_architecture if exactly one does; "
                    "route_closed if both read no_gap; inconclusive otherwise"),
        "route_open": ("pruned detectors miss unseen attack families that the dense detector catches, at matched in-distribution risk, "
                       "and recovery on training data cannot close it; the method paper targets this gap with a recovery objective "
                       "that preserves the teacher's behaviour off the training distribution; next is the design of that objective "
                       "and its baselines, not more measurement"),
        "route_open_one_architecture": "as route_open, scoped to that architecture, with the other architecture reported as a boundary of the claim",
        "route_closed": ("compression does not make these detectors forget unseen attack families; the off-distribution method route "
                         "is closed on this setup, and method effort moves to a problem that is open"),
        "inconclusive": ("the cells are reported as they are; no method is built on a partial result; a targeted follow-up (more seeds "
                         "or families) is run only if a cell is close to the mark"),
    },
    "budgets": [0.70, 0.90], "primary_budget": 0.90, "methods": ["magnitude", "fisher"],
    "gap_threshold": 0.05, "match_threshold": 0.02, "teacher_epochs": 4, "recovery_epochs": 2,
    "no_test_access": True,
}
_text = json.dumps(PREREG, indent=2)
PRE = OUT / "N41_PREREGISTRATION.json"
if PRE.exists() and PRE.read_text() != _text:
    raise RuntimeError("N41_PREREGISTRATION.json already exists with different content. The pre-registration is frozen; restore this cell to the committed version.")
PRE.write_text(_text); print(_text)

ARCHS = ["shallow", "deep"]; FAMILIES = PREREG["design"]["held_out_families"]; SEEDS = PREREG["design"]["seeds"]
METHODS = PREREG["methods"]; BUDGETS = PREREG["budgets"]; PRIMARY = PREREG["primary_budget"]
GAP = PREREG["gap_threshold"]; MATCH = PREREG["match_threshold"]; T_EPOCHS = PREREG["teacher_epochs"]; R_EPOCHS = PREREG["recovery_epochs"]
TOL = 0.015; CAL_SEED = 2026; CLIP_SEED = 2026; FISHER_BATCHES = 40
MIN_W = {"shallow": int(yaml.safe_load(open(REPO / "config/saber.yaml"))["groups"]["minimum_remaining_per_layer"]), "deep": 8}
ACCEPT = {"b2a_max": 0.10, "attack_recall_min": 0.85, "fine_f1_min": 0.40}
print("per-layer floor:", MIN_W)

In [ ]:
# Stage 2b - commit the pre-registration before any result exists (skipped automatically once results exist)
def _prereg_committed():
    rel = "results/saber/41_unseen_family_compression/N41_PREREGISTRATION.json"
    h = subprocess.run(["git", "log", "-1", "--format=%h", "--", rel], capture_output=True, text=True, cwd=REPO).stdout.strip()
    dirty = subprocess.run(["git", "status", "--porcelain", "--", rel], capture_output=True, text=True, cwd=REPO).stdout.strip()
    return bool(h) and not dirty


_results_exist = any(p.is_file() and p.name != "N41_PREREGISTRATION.json" for p in OUT.rglob("*"))
if _results_exist:
    assert _prereg_committed(), ("results exist but the pre-registration is not committed, so it can no longer be committed before any "
                                 "result; stop here and report it")
    print("results already exist in", OUT.name, "and the pre-registration is committed; nothing to do here")
else:
    os.chdir(REPO)
    r = subprocess.run(["python", "tools/commit_cell.py",
                        "NB41 pre-registration, committed before any result: do pruned and recovered CICIoT2023 detectors miss attack "
                        "families they never saw? Three families held out in turn (dos, reconnaissance, web_application); per cell a dense "
                        "teacher trained from scratch without the family (4 epochs), a dense reference with two further epochs, and four "
                        "students (magnitude and Fisher at 70% and 90% realised FLOPs, two full epochs of recovery), all clipped and "
                        "recalibrated; both networks, 5 seeds; primary endpoint the fraction of the unseen family raised as an attack, "
                        "paired against the dense reference; P1 gap of 0.05 at matched in-distribution risk; readings route_open, "
                        "route_open_one_architecture, route_closed, inconclusive."],
                       capture_output=True, text=True)
    print(r.stdout); print(r.stderr)
    assert _prereg_committed(), "the pre-registration commit did not go through; fix it before running stage 3"
    print("pre-registration committed:", subprocess.run(["git", "log", "-1", "--format=%h %s"], capture_output=True, text=True, cwd=REPO).stdout.strip()[:120])

In [ ]:
# Stage 3 - data and helpers
TRAIN_LOADER, VAL_LOADER, _T, _SHALLOW_UNUSED, CLASS_NAMES = load_bridge()
FULL_TAX = ciciot2023_taxonomy(CLASS_NAMES); N_CLASSES = len(CLASS_NAMES); FAM_ALL = np.array(FULL_TAX.family_by_class)
assert set(FAMILIES) <= set(FAM_ALL.tolist()), f"unknown family in {FAMILIES}"
Xt, Yt = TRAIN_LOADER.dataset.tensors; yt = Yt.numpy(); N_FEAT = int(Xt.shape[1])
Xv, Yv = VAL_LOADER.dataset.tensors; VAL_Y_ALL = Yv.numpy()
_rng = np.random.default_rng(0)
_idx = np.concatenate([_rng.permutation(np.where(VAL_Y_ALL == c)[0])[:4000] for c in range(N_CLASSES) if (VAL_Y_ALL == c).sum() > 0])
_idx = np.random.default_rng(12345).permutation(_idx)
EX_X_ALL = Xv[_idx]; EX_Y_ALL = VAL_Y_ALL[_idx]
del Xv, Yv
print("train rows", len(yt), "| eval rows", len(_idx), "| features", N_FEAT, "| families", sorted(set(FAM_ALL.tolist())))


class Clipped(nn.Module):
    def __init__(self, inner, lo, hi):
        super().__init__(); self.inner = inner; self.register_buffer("lo", lo.clone()); self.register_buffer("hi", hi.clone())
    def forward(self, x):
        return self.inner(torch.maximum(torch.minimum(x, self.hi), self.lo))


class Setting:
    """Everything that depends on which family is held out: rows, labels, taxonomy, weights, clip bounds, calibration slice."""
    def __init__(self, family):
        self.family = family
        self.seen = [c for c in range(N_CLASSES) if FAM_ALL[c] != family]; self.unseen = [c for c in range(N_CLASSES) if FAM_ALL[c] == family]
        assert self.unseen, f"family {family} has no classes"
        self.remap = {c: i for i, c in enumerate(self.seen)}; self.names = [CLASS_NAMES[c] for c in self.seen]; self.K = len(self.seen)
        self.tax = ciciot2023_taxonomy(self.names); self.fam = np.array(self.tax.family_by_class); self.benign = self.names.index(self.tax.benign_class)
        self.seen_families = sorted(set(self.fam.tolist()))
        keep = np.isin(yt, self.seen); self.train_idx = np.where(keep)[0]
        self.y_train = np.array([self.remap[c] for c in yt[keep]], dtype=np.int64); self.n_train = len(self.train_idx)
        self.ds = torch.utils.data.TensorDataset(Xt[torch.from_numpy(self.train_idx)], torch.from_numpy(self.y_train))
        counts = np.bincount(self.y_train, minlength=self.K); w = np.zeros(self.K); w[counts > 0] = 1 / np.sqrt(counts[counts > 0]); w[counts > 0] /= w[counts > 0].mean()
        self.class_w = torch.tensor(w, dtype=torch.float32, device=DEVICE)
        sub = self.ds.tensors[0][torch.randperm(self.n_train, generator=torch.Generator().manual_seed(CLIP_SEED))[:400_000]].numpy()
        self.lo = torch.tensor(np.percentile(sub, 0.1, axis=0), dtype=torch.float32, device=DEVICE); self.hi = torch.tensor(np.percentile(sub, 99.9, axis=0), dtype=torch.float32, device=DEVICE)
        cal = torch.randperm(self.n_train, generator=torch.Generator().manual_seed(CAL_SEED))[:50 * 1024]
        self.cal_loader = torch.utils.data.DataLoader(torch.utils.data.Subset(self.ds, cal.tolist()), batch_size=1024, shuffle=False)
        m_seen = np.isin(EX_Y_ALL, self.seen); m_un = np.isin(EX_Y_ALL, self.unseen)
        self.ex_x = EX_X_ALL[torch.from_numpy(m_seen)].to(DEVICE); self.ex_y = np.array([self.remap[c] for c in EX_Y_ALL[m_seen]], dtype=np.int64)
        self.un_x = EX_X_ALL[torch.from_numpy(m_un)].to(DEVICE); self.un_y = EX_Y_ALL[m_un]
        self.example = self.ex_x[:8].float()
        self.fam_y = self.fam[self.ex_y]

    def loader(self, seed):
        return torch.utils.data.DataLoader(self.ds, batch_size=1024, shuffle=True, generator=torch.Generator().manual_seed(seed))

    def make(self, arch):
        return M.build("cnn1d", in_dim=N_FEAT, n_classes=self.K, channels=(64, 128)) if arch == "shallow" else DeepCNN1D(self.K)


def forward_logits(model, X):
    model.eval()
    with torch.no_grad():
        return torch.cat([model(X[i:i + 8192]).cpu() for i in range(0, len(X), 8192)]).numpy()


def train_epochs(model, loader, epochs, weights):
    opt = torch.optim.Adam(model.parameters(), lr=1e-3); lossf = nn.CrossEntropyLoss(weight=weights)
    for _ in range(epochs):
        model.train()
        for xb, yb in loader:
            opt.zero_grad(); lossf(model(xb.to(DEVICE)), yb.to(DEVICE)).backward(); opt.step()
    return model.eval()


def recalibrate_bn(model, loader, n_batches=50):
    for m in model.modules():
        if isinstance(m, nn.BatchNorm1d): m.reset_running_stats(); m.momentum = None
    model.train()
    with torch.no_grad():
        for i, (xb, _) in enumerate(loader):
            if i >= n_batches: break
            model(xb.to(DEVICE))
    for m in model.modules():
        if isinstance(m, nn.BatchNorm1d): m.momentum = 0.1
    model.eval(); return model


def softmax_np(z):
    z = z - z.max(axis=1, keepdims=True); e = np.exp(z); return e / e.sum(axis=1, keepdims=True)


def audit(model, S):
    """In-distribution audit on the remaining classes, plus the unseen-family audit."""
    lg = forward_logits(model, S.ex_x); a = full_model_audit(lg, S.ex_y, S.tax, DEFAULT_COST_PROFILES); pf = S.fam[lg.argmax(axis=1)]
    out = {"seen_hsr_balanced": float(a["hsr_balanced_soc"]), "seen_hsr_miss": float(a["hsr_miss_sensitive"]), "seen_hsr_fatigue": float(a["hsr_alert_fatigue"]),
           "seen_b2a": float(a["benign_to_attack_rate"]), "seen_a2b": float(a["attack_to_benign_rate"]), "seen_attack_recall": float(a["binary_attack_recall"]),
           "seen_family_f1": float(a["family_macro_f1"]), "seen_fine_f1": float(a["fine_macro_f1"]), "seen_ece": float(a["ece15"]),
           "seen_maxprob": float(softmax_np(lg).max(axis=1).mean())}
    out.update({f"seen_recall_{f}": float((pf[S.fam_y == f] == f).mean()) for f in S.seen_families})
    lu = forward_logits(model, S.un_x); pu = lu.argmax(axis=1); p = softmax_np(lu); fu = S.fam[pu]
    out.update({"unseen_rows": int(len(pu)), "unseen_detect": float((pu != S.benign).mean()), "unseen_attack_mass": float((1 - p[:, S.benign]).mean()),
                "unseen_maxprob": float(p.max(axis=1).mean())})
    out.update({f"absorbed_{f}": float((fu == f).mean()) for f in S.seen_families})
    return out


def accepted(row):
    return bool(row["seen_b2a"] <= ACCEPT["b2a_max"] and row["seen_attack_recall"] >= ACCEPT["attack_recall_min"] and row["seen_fine_f1"] >= ACCEPT["fine_f1_min"])


def valid_order(table, column, min_w):
    t = table.sort_values(column, ascending=True); remaining = t.groupby("module_path")["group_id"].count().to_dict(); order = []
    for row in t.itertuples():
        layer = str(row.module_path)
        if remaining[layer] - 1 < min_w: continue
        remaining[layer] -= 1; order.append((layer, int(row.channel_index)))
    return order


def prune_map_of(pairs):
    pm = {}
    for layer, ch in pairs: pm.setdefault(str(layer), []).append(int(ch))
    return {l: sorted(c) for l, c in pm.items()}


def widths(st):
    return "/".join(str(m.out_channels) for _, m in st.named_modules() if isinstance(m, nn.Conv1d))


SETTINGS = {f: Setting(f) for f in FAMILIES}
for f, S in SETTINGS.items():
    print(f"held out {f:16s}: classes {len(S.unseen)} | remaining classes {S.K} | train rows {S.n_train:,} | eval rows seen {len(S.ex_y):,} unseen {len(S.un_y):,}")

In [ ]:
# Stage 4 - the runs: per (architecture, family, seed) one teacher, one dense reference, four students (resumable; teachers cached)
RUNS = OUT / "N41_runs.csv"
rows = pd.read_csv(RUNS).to_dict("records") if RUNS.exists() else []
done = {(r["architecture"], r["family"], int(r["seed"]), r["model"]) for r in rows}
MODELS = ["teacher_raw", "dense"] + [f"{m}_b{int(round(b * 100))}" for m in METHODS for b in BUDGETS]
struct_path = OUT / "N41_structures.csv"; struct_rows = pd.read_csv(struct_path).to_dict("records") if struct_path.exists() else []
(OUT / "structures").mkdir(exist_ok=True)


def save_row(row):
    rows.append(row); done.add((row["architecture"], row["family"], int(row["seed"]), row["model"])); pd.DataFrame(rows).to_csv(RUNS, index=False)
    print(f"  {row['model']:14s}: unseen detect {row['unseen_detect']:.3f} (attack mass {row['unseen_attack_mass']:.3f}) | seen HSR {row['seen_hsr_balanced']:.4f} "
          f"b2a {row['seen_b2a']:.4f} a2b {row['seen_a2b']:.4f} famF1 {row['seen_family_f1']:.3f} | {row['seconds']:.0f}s")


def teacher_for(arch, S, seed):
    ck = TEACHER_DIR / f"{arch}_{S.family}_s{seed}.pt"; inner = S.make(arch)
    if ck.exists():
        inner.load_state_dict(torch.load(ck, map_location="cpu", weights_only=False)["state_dict"]); return Clipped(inner, S.lo, S.hi).to(DEVICE).eval(), True
    torch.manual_seed(seed); np.random.seed(seed); inner = S.make(arch)
    model = Clipped(inner, S.lo, S.hi).to(DEVICE); train_epochs(model, S.loader(seed), T_EPOCHS, S.class_w)
    torch.save({"state_dict": model.inner.cpu().state_dict(), "architecture": arch, "family": S.family, "seed": seed, "classes": S.names}, ck)
    model.inner.to(DEVICE); return model.eval(), False


for arch, family, seed in itertools.product(ARCHS, FAMILIES, SEEDS):
    if all((arch, family, seed, m) in done for m in MODELS): continue
    S = SETTINGS[family]; base = {"architecture": arch, "family": family, "seed": seed}
    print(f"\n== {arch} | held out {family} | seed {seed}")
    t0 = time.time(); teacher, cached = teacher_for(arch, S, seed)
    if (arch, family, seed, "teacher_raw") not in done:
        tr = copy.deepcopy(teacher); recalibrate_bn(tr, S.cal_loader); a = audit(tr, S)
        save_row({**base, "model": "teacher_raw", "method": "dense", "budget": 0.0, "parameters": int(count_parameters(teacher.inner)), "realised_reduction": 0.0,
                  "teacher_accepted": accepted(a), "teacher_cached": cached, "seconds": round(time.time() - t0, 1), **a}); del tr
    teacher_row = next(r for r in rows if (r["architecture"], r["family"], int(r["seed"]), r["model"]) == (arch, family, seed, "teacher_raw"))
    acc = bool(teacher_row["teacher_accepted"])
    if not acc: print("  teacher FAILS the acceptance rule; its cell is excluded from the verdict and listed")
    if (arch, family, seed, "dense") not in done:
        t0 = time.time(); torch.manual_seed(seed + 1); dense = copy.deepcopy(teacher); train_epochs(dense, S.loader(seed + 1), R_EPOCHS, S.class_w); recalibrate_bn(dense, S.cal_loader)
        save_row({**base, "model": "dense", "method": "dense", "budget": 0.0, "parameters": int(count_parameters(dense.inner)), "realised_reduction": 0.0,
                  "teacher_accepted": acc, "teacher_cached": cached, "seconds": round(time.time() - t0, 1), **audit(dense, S)}); del dense
    # scores on the teacher (remaining-class training batches, clipped inputs), removal orders, calibrated structures
    groups = enumerate_cnn1d_channel_groups(teacher.inner, S.example)
    mag = magnitude_scores(teacher.inner, groups)
    grad = gradient_saliency_scores(teacher.inner, S.loader(seed + 2), groups, device=DEVICE, max_batches=FISHER_BATCHES, criterion=nn.CrossEntropyLoss(weight=S.class_w),
                                    input_transform=lambda x: torch.maximum(torch.minimum(x, S.hi), S.lo))
    for t in (groups, mag, grad): t["group_id"] = t["group_id"].astype(str)
    table = groups[["group_id", "module_path", "channel_index"]].merge(mag, on="group_id", validate="1:1").merge(grad[["group_id", "fisher"]], on="group_id", validate="1:1")
    m0 = profile_forward_flops(teacher.inner, S.example)["flops_per_item"]
    for method in METHODS:
        order = valid_order(table, method, MIN_W[arch])
        build = lambda k, o=order: prune_cnn1d_channels(teacher.inner, prune_map_of(o[:k]), S.example, minimum_remaining_per_layer=MIN_W[arch])[0].to(DEVICE)
        realised = lambda st: 1.0 - profile_forward_flops(st, S.example)["flops_per_item"] / m0
        for budget in BUDGETS:
            name = f"{method}_b{int(round(budget * 100))}"
            if (arch, family, seed, name) in done: continue
            t0 = time.time(); cal = calibrate_prefix_count(order, target_reduction=budget, build_model=build, realised_reduction=realised, tolerance=TOL)
            assert cal.within_tolerance, f"{arch}/{family}/s{seed}/{name}: realised {cal.realised_reduction:.4f} is outside the tolerance"
            torch.manual_seed(seed + 1); st = Clipped(build(cal.prefix_length), S.lo, S.hi).to(DEVICE)
            struct_rows = [r for r in struct_rows if (r["architecture"], r["family"], int(r["seed"]), r["model"]) != (arch, family, seed, name)]
            struct_rows.append({**base, "model": name, "k": cal.prefix_length, "realised_reduction": cal.realised_reduction, "widths": widths(st.inner), "parameters": int(count_parameters(st.inner))})
            pd.DataFrame(struct_rows).to_csv(struct_path, index=False)
            pd.DataFrame([{"module_path": l, "channel_index": c} for l, c in order[:cal.prefix_length]]).to_csv(OUT / f"structures/N41_{arch}_{family}_s{seed}_{name}_removed_groups.csv", index=False)
            train_epochs(st, S.loader(seed + 1), R_EPOCHS, S.class_w); recalibrate_bn(st, S.cal_loader)
            save_row({**base, "model": name, "method": method, "budget": budget, "parameters": int(count_parameters(st.inner)), "realised_reduction": cal.realised_reduction,
                      "teacher_accepted": acc, "teacher_cached": cached, "seconds": round(time.time() - t0, 1), **audit(st, S)}); del st
    del teacher; gc.collect(); torch.cuda.empty_cache() if DEVICE == "cuda" else None
print("\nrows:", len(rows), "of", len(ARCHS) * len(FAMILIES) * len(SEEDS) * len(MODELS))

In [ ]:
# Stage 5 - analysis and verdict (rules exactly as pre-registered in stage 2)
runs = pd.read_csv(RUNS)
EXPECTED = len(ARCHS) * len(FAMILIES) * len(SEEDS) * len(MODELS)
assert len(runs) == EXPECTED, f"expected {EXPECTED} rows, found {len(runs)}: finish stage 4 first"
runs["teacher_accepted"] = runs["teacher_accepted"].astype(bool)
excluded = runs[(runs.model == "teacher_raw") & (~runs.teacher_accepted)][["architecture", "family", "seed", "seen_b2a", "seen_attack_recall", "seen_fine_f1"]]
ok = runs[runs.teacher_accepted]


def paired(arch, family, model, col):
    d = ok[(ok.architecture == arch) & (ok.family == family) & (ok.model == "dense")].set_index("seed")[col]
    s = ok[(ok.architecture == arch) & (ok.family == family) & (ok.model == model)].set_index("seed")[col]
    common = d.index.intersection(s.index); g = (d.loc[common] - s.loc[common]).to_numpy(); n = len(g)
    if n < 3: return {"n": int(n), "gap": float("nan"), "gap_lo": float("nan"), "gap_hi": float("nan")}
    hw = stats.t.ppf(0.975, n - 1) * g.std(ddof=1) / np.sqrt(n)
    return {"n": int(n), "gap": float(g.mean()), "gap_lo": float(g.mean() - hw), "gap_hi": float(g.mean() + hw)}


cells = []
for arch, family, method, budget in itertools.product(ARCHS, FAMILIES, METHODS, BUDGETS):
    model = f"{method}_b{int(round(budget * 100))}"; pg = paired(arch, family, model, "unseen_detect"); ph = paired(arch, family, model, "seen_hsr_balanced")
    dense = ok[(ok.architecture == arch) & (ok.family == family) & (ok.model == "dense")]; stud = ok[(ok.architecture == arch) & (ok.family == family) & (ok.model == model)]
    cells.append({"architecture": arch, "family": family, "method": method, "budget": budget, "n": pg["n"],
                  "dense_unseen_detect": float(dense.unseen_detect.mean()), "student_unseen_detect": float(stud.unseen_detect.mean()),
                  "gap": pg["gap"], "gap_lo": pg["gap_lo"], "gap_hi": pg["gap_hi"], "gap_condition": bool(pg["n"] >= 3 and pg["gap"] >= GAP and pg["gap_lo"] > 0),
                  "student_better": bool(pg["n"] >= 3 and pg["gap_hi"] < 0),
                  "hsr_diff_student_minus_dense": (-ph["gap"] if ph["n"] >= 3 else float("nan")), "matched": bool(ph["n"] >= 3 and abs(ph["gap"]) <= MATCH),
                  "dense_unseen_attack_mass": float(dense.unseen_attack_mass.mean()), "student_unseen_attack_mass": float(stud.unseen_attack_mass.mean()),
                  "dense_absorbed_benign": float(dense.absorbed_benign.mean()), "student_absorbed_benign": float(stud.absorbed_benign.mean()),
                  "student_parameters": float(stud.parameters.mean())})
CELLS = pd.DataFrame(cells); CELLS.to_csv(OUT / "N41_cells.csv", index=False)

H = {}
for arch in ARCHS:
    c90 = CELLS[(CELLS.architecture == arch) & np.isclose(CELLS.budget, PRIMARY)]
    fam_gap = {f: bool(all(c90[(c90.family == f) & (c90.method == m)].gap_condition.iloc[0] for m in METHODS)) for f in FAMILIES}
    fam_ok = {f: bool(fam_gap[f] and all(c90[(c90.family == f) & (c90.method == m)].matched.iloc[0] for m in METHODS)) for f in FAMILIES}
    P1 = bool(sum(fam_ok.values()) >= 2)
    if P1: reading = "gap_at_matched_risk"
    elif sum(fam_gap.values()) >= 2: reading = "gap_but_unmatched"
    elif c90.gap_condition.any(): reading = "partial"
    else: reading = "no_gap"
    student_better = bool(c90.student_better.sum() >= len(c90) / 2)
    dose = []
    for f, m in itertools.product(FAMILIES, METHODS):
        g90 = CELLS[(CELLS.architecture == arch) & (CELLS.family == f) & (CELLS.method == m) & np.isclose(CELLS.budget, 0.90)].gap.iloc[0]
        g70 = CELLS[(CELLS.architecture == arch) & (CELLS.family == f) & (CELLS.method == m) & np.isclose(CELLS.budget, 0.70)].gap.iloc[0]
        dose.append(bool(g90 > g70))
    H[arch] = {"P1_gap": P1, "reading": reading, "student_better_flag": student_better, "families_meeting_gap": fam_gap, "families_meeting_gap_and_matched": fam_ok,
               "S1_dose_cells_90_above_70": f"{sum(dose)} of {len(dose)}",
               "S2_dense_unseen_detect": {f: float(c90[c90.family == f].dense_unseen_detect.iloc[0]) for f in FAMILIES},
               "cells_90": json.loads(c90.round(4).to_json(orient="records"))}
r_ = [H[a]["reading"] for a in ARCHS]
overall = ("route_open" if all(x == "gap_at_matched_risk" for x in r_) else "route_open_one_architecture" if sum(x == "gap_at_matched_risk" for x in r_) == 1
           else "route_closed" if all(x == "no_gap" for x in r_) else "inconclusive")
verdict = {"notebook": "41_unseen_family_compression", "reading": overall, "meaning": PREREG["readings"][overall], "by_architecture": H,
           "excluded_cells_teacher_failed_acceptance": json.loads(excluded.round(4).to_json(orient="records")), "prereg": PREREG}
(OUT / "N41_verdict.json").write_text(json.dumps(verdict, indent=2, default=float))

print("OVERALL READING:", overall, "-", PREREG["readings"][overall])
print("cells excluded (teacher failed acceptance):", len(excluded))
for arch in ARCHS:
    h = H[arch]
    print(f"\n{arch}: P1 {h['P1_gap']} -> {h['reading']} | student-better flag {h['student_better_flag']} | dose 90%>70% in {h['S1_dose_cells_90_above_70']} cells")
    print("  dense unseen detection (ceiling):", {f: round(v, 3) for f, v in h["S2_dense_unseen_detect"].items()})
show = CELLS[["architecture", "family", "method", "budget", "n", "dense_unseen_detect", "student_unseen_detect", "gap", "gap_lo", "gap_hi", "gap_condition", "matched",
              "hsr_diff_student_minus_dense", "student_absorbed_benign", "dense_absorbed_benign"]]
print(show.round(4).to_string(index=False))

In [ ]:
# Stage 6 - figure (600 dpi, paper names)
runs = pd.read_csv(RUNS); runs["teacher_accepted"] = runs["teacher_accepted"].astype(bool); ok = runs[runs.teacher_accepted]
LABEL = {"dense": "Dense detector (recovered)", "magnitude_b70": "Magnitude pruning, 70%", "magnitude_b90": "Magnitude pruning, 90%",
         "fisher_b70": "Fisher pruning, 70%", "fisher_b90": "Fisher pruning, 90%"}
FAMILY_LABEL = {"dos": "DoS", "reconnaissance": "Reconnaissance", "web_application": "Web application", "ddos": "DDoS", "mirai": "Mirai",
                "spoofing_mitm": "Spoofing / MITM", "brute_force": "Brute force", "malware": "Malware", "benign": "Benign"}
ARCH_LABEL = {"shallow": "Shallow CNN (64/128)", "deep": "Deep CNN (64/128/128/256)"}
order = ["dense", "magnitude_b70", "magnitude_b90", "fisher_b70", "fisher_b90"]
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for j, arch in enumerate(ARCHS):
    for i, (col, ylabel) in enumerate([("unseen_detect", "Unseen family raised as an attack (fraction)"), ("seen_hsr_balanced", "In-distribution risk (HSR, balanced profile)")]):
        ax = axes[i, j]; x = np.arange(len(FAMILIES)); w = 0.16
        for k, model in enumerate(order):
            q = ok[(ok.architecture == arch) & (ok.model == model)].groupby("family")[col]
            means = [float(q.mean().get(f, np.nan)) for f in FAMILIES]; sds = [float(q.std(ddof=1).get(f, np.nan)) for f in FAMILIES]
            ax.bar(x + (k - 2) * w, means, w, yerr=sds, capsize=2, label=LABEL[model], color=("0.35" if model == "dense" else None))
        ax.set_xticks(x); ax.set_xticklabels([FAMILY_LABEL.get(f, f) for f in FAMILIES]); ax.set_ylabel(ylabel)
        ax.set_title(f"{ARCH_LABEL[arch]}: {'held-out family' if i == 0 else 'remaining classes'}", fontsize=10)
        if i == 0: ax.set_ylim(0, 1.0)
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=5, fontsize=8, frameon=False)
fig.suptitle("Detection of an attack family absent from training: dense detector against its pruned and recovered copies (mean and SD over seeds)", fontsize=10)
fig.tight_layout(rect=[0, 0.06, 1, 0.97]); fig.savefig(OUT / "N41_unseen_family.png", dpi=600); plt.show(); print("written ->", OUT)

In [ ]:
# Stage 7 - commit the results (standard commit cell; the message is filled in from the verdict)
v = json.load(open(OUT / "N41_verdict.json"))
pre = subprocess.run(["git", "log", "-1", "--format=%h", "--", "results/saber/41_unseen_family_compression/N41_PREREGISTRATION.json"], capture_output=True, text=True, cwd=REPO).stdout.strip() or "see log"
parts = []
for arch in ARCHS:
    h = v["by_architecture"][arch]; ceiling = ", ".join(f"{f} {x:.2f}" for f, x in h["S2_dense_unseen_detect"].items())
    g = lambda x: float("nan") if x is None else float(x)
    gaps = "; ".join(f"{c['family']}/{c['method']} gap {g(c['gap']):+.3f} [{g(c['gap_lo']):+.3f}, {g(c['gap_hi']):+.3f}]{'' if c['matched'] else ' (unmatched)'}"
                     + (f" (n={c['n']}, no interval)" if c["n"] < 3 else "") for c in h["cells_90"])
    parts.append(f"{arch}: P1 {'HELD' if h['P1_gap'] else 'FAILED'} -> {h['reading']}; dense unseen detection {ceiling}; 90% cells: {gaps}; dose 90%>70% in {h['S1_dose_cells_90_above_70']} cells.")
msg = (f"NB41 results (pre-registration {pre}): unseen attack families under compression on CICIoT2023, overall reading {v['reading']}. " + " ".join(parts)
       + f" Cells excluded for a teacher failing the acceptance rule: {len(v['excluded_cells_teacher_failed_acceptance'])}.")
print(msg)
os.chdir(REPO)
r = subprocess.run(["python", "tools/commit_cell.py", msg], capture_output=True, text=True)
print(r.stdout); print(r.stderr)